# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1: The model state the SEO of the sites.
My methodology question: Where does the label in this finding come from, it is itindependently measured, or could it be derived from one of the same signals used to predict it.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

!git clone https://github.com/muhammadhuzaifanaeem/FlyRank-Internship.git


df = pd.read_csv('FlyRank-Internship/data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

feature_cols = ['impressions_90d', 'clicks_90d', 'sessions_90d', 'avg_position', 'ctr',
                 'engagement_rate', 'scroll_rate', 'content_age_days',
                 'days_since_last_update', 'word_count']

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# --- "BEFORE": a plain random split (the dishonest baseline to expose) ---
train_r, test_r = train_test_split(df, test_size=0.25, random_state=42)
X_train_r, y_train_r = train_r[feature_cols].fillna(0), train_r['is_declining_label']
X_test_r, y_test_r = test_r[feature_cols].fillna(0), test_r['is_declining_label']

rf_random = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)
rf_random.fit(X_train_r, y_train_r)
scores_random = rf_random.predict_proba(X_test_r)[:, 1]

# --- "AFTER": grouped split by client (the honest version) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_id']))
train_g, test_g = df.iloc[train_idx], df.iloc[test_idx]
X_train_g, y_train_g = train_g[feature_cols].fillna(0), train_g['is_declining_label']
X_test_g, y_test_g = test_g[feature_cols].fillna(0), test_g['is_declining_label']

rf_grouped = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)
rf_grouped.fit(X_train_g, y_train_g)
scores_grouped = rf_grouped.predict_proba(X_test_g)[:, 1]

# --- Compare BEFORE vs AFTER ---
comparison = pd.DataFrame({
    'Split type': ['Random split (before)', 'Grouped by client (after)'],
    'Base rate': [y_test_r.mean(), y_test_g.mean()],
    'Precision@20': [precision_at_k(scores_random, y_test_r, 20), precision_at_k(scores_grouped, y_test_g, 20)],
    'Precision@50': [precision_at_k(scores_random, y_test_r, 50), precision_at_k(scores_grouped, y_test_g, 50)],
})
print(comparison.round(3))

Cloning into 'FlyRank-Internship'...
remote: Enumerating objects: 115, done.
remote: Counting objects: 100% (115/115), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 115 (delta 29), reused 80 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (115/115), 1.83 MiB | 15.36 MiB/s, done.
Resolving deltas: 100% (29/29), done.
                  Split type  Base rate  Precision@20  Precision@50
0      Random split (before)      0.546           1.0          0.98
1  Grouped by client (after)      0.517           0.5          0.58


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [7]:
# --- Check 1: label-derived features (train with vs without suspects) ---
suspect_cols = [c for c in feature_cols if 'trend' in c.lower() or 'decline' in c.lower()]
print("Suspicious label-derived columns found in features:", suspect_cols if suspect_cols else "NONE — clean")

# --- Check 2: feature importance sanity check (does one feature tower over the rest?) ---
importances = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_grouped.feature_importances_
}).sort_values('importance', ascending=False)
print("\nFeature importances:")
print(importances)

top_share = importances['importance'].iloc[0]
print(f"\nTop feature share of total importance: {top_share:.2%}",
      "— investigate if this looks suspiciously dominant (e.g. above ~50%)" if top_share > 0.5 else "— looks reasonably distributed")

# --- Check 3: the "train without" test (the confession test from the skill file) ---
if suspect_cols:
    reduced_features = [c for c in feature_cols if c not in suspect_cols]
    rf_reduced = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)
    rf_reduced.fit(train_g[reduced_features].fillna(0), y_train_g)
    scores_reduced = rf_reduced.predict_proba(test_g[reduced_features].fillna(0))[:, 1]
    print("\nPrecision@50 WITH suspect features:", round(precision_at_k(scores_grouped, y_test_g, 50), 3))
    print("Precision@50 WITHOUT suspect features:", round(precision_at_k(scores_reduced, y_test_g, 50), 3))
else:
    print("\nNo suspect features found — 'train without' test not needed.")

Suspicious label-derived columns found in features: NONE — clean

Feature importances:
                  feature  importance
0         impressions_90d    0.277645
3            avg_position    0.203986
7        content_age_days    0.161640
9              word_count    0.104443
6             scroll_rate    0.055250
4                     ctr    0.050993
8  days_since_last_update    0.048135
1              clicks_90d    0.045486
2            sessions_90d    0.037275
5         engagement_rate    0.015146

Top feature share of total importance: 27.76% — looks reasonably distributed

No suspect features found — 'train without' test not needed.


I checked my final feature set against all three leakage patterns: label-derived columns, future-window overlaps, and product-flag circularity. [State your actual findings — e.g., "No suspect columns were found by name, and feature importance was reasonably distributed with no single feature dominating."]

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Before (too bold): "My model proves which pages will decline and should be refreshed."

After (honest): "On this held-out set of clients, my model ranks pages by likelihood of decline at a precision@50 of meaning these pages look worth reviewing first, based on patterns observed in this data. This is decision-support, not a guarantee, and hasn't been tested against a live intervention."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.